In [2]:
# !pip install -U pip wheel
# !pip install -q -r /Users/ryan/github/lltk/requirements.txt
import sys
sys.path.insert(0,'..')
sys.path.insert(0,'/Users/ryan/github/prosodic')
sys.path.insert(0,'/Users/ryan/github/lltk')
sys.path.insert(0,'/Users/ryan/github/logmap')
from llmdh import *
import prosodic
from prosodic import Word
prosodic.USE_CACHE=False
import plotnine as p9
p9.options.figure_size=11,8

In [3]:
def get_df_poems(min_lines=8):
    df_poems = pd.read_pickle('data.allpoems.pkl').fillna('')
    
    excl_prompts=[
        'Write an unryhmed poem in the style of Shakespeare\'s dramatic monologues.',
        'Write a poem in the style of Shakespeare\'s dramatic monologues.',
        'Write a poem in the style of e.e. cummings',
        # 'Write a poem in the style of Walt Whitman.',
        'Write a poem in the style of Wallace Stevens.',
        'Continue the following poem:\n\nTyping, typing, fingers on the keyboard\nThe keys crack and bend under sweat and weight,\n'
    ]

    df_poems = df_poems[~df_poems.prompt.isin(excl_prompts)]
    df_poems['num_lines'] = pd.to_numeric(df_poems['num_lines'], errors='coerce')
    return df_poems.query(f'num_lines >= {min_lines}')

In [4]:
df_poems = get_df_poems()
# df_poems

In [5]:
def get_txt_rhyming_data(txt, max_dist=0):
    bad_openings = ['Here is', 'Here\'s a']
    lines1 = txt.split('\n')
    lines = [x for x in lines1 if not any(x.startswith(y) for y in bad_openings)]
    txt = '\n'.join(lines).strip()
    poem = prosodic.Text(txt=txt)
    data = {'poem':txt}
    data['num_lines'] = poem.num_lines
    if data['num_lines']:
        rhymes = poem.get_rhyming_lines(max_dist=max_dist)
        rhymeset = set(rhymes.keys()) | set(rhymes.values())
        data['num_rhyming_lines'] = len(rhymeset)
        assert data['num_rhyming_lines'] <= data['num_lines']
    else:
        data['num_rhyming_lines'] = np.nan
    return data

In [6]:
# poem_txt = df_poems.query('model == "b. 1950-2000"').sample(n=1).iloc[0].poem
# print(poem_txt)
# poem = prosodic.Text(poem_txt)
# pprint(poem.get_rhyming_lines(max_dist=0))
# get_txt_rhyming_data(poem_txt)

In [7]:
def get_rhyme_data(fn='data.allpoems.pkl', force=False, min_lines=10, lim=None):
    df = get_df_poems(min_lines=min_lines)
    df['poem_hash'] = df['poem'].apply(hashstr)
    df = df.drop_duplicates('poem_hash')
    df = df.groupby(['model','prompt']).sample(n=10000,replace=True)
    df = df.drop_duplicates('poem_hash')

    ofn = os.path.splitext(fn)[0]+'.rhyme_data3.tsv'
    done = set()
    if os.path.exists(ofn):
        with open(ofn) as f:
            for ln in f:
                id = ln.strip().split('\t')[0]
                if id:
                    done.add(id)
    print('Done already:',len(done))

    df = df[~df.poem_hash.isin(done)]
    poems = df.sample(frac=1).poem.drop_duplicates()

    col = ['poem', 'num_lines', 'num_rhyming_lines']
    file_exists = os.path.exists(ofn)
    with open(ofn,'a+') as of:
        if not file_exists:
            of.write('\t'.join(col)+'\n')
        for poem in tqdm(poems[:lim]):
            try:
                data = get_txt_rhyming_data(poem)
            except Exception:
                continue
            data['poem'] = hashstr(data['poem'])
            outstr = '\t'.join(str(data[k]) for k in col) + '\n'
            of.write(outstr)
    return ofn

In [8]:
get_rhyme_data()

Done already: 15490


 31%|███▏      | 11499/36661 [3:54:04<30:18:56,  4.34s/it]   

: 